In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)

In [ ]:
# Load data and labels
data_dir = Path("../data/UNSW-NB15")

df = pd.read_csv(data_dir / "Data.csv", low_memory=False)
labels = pd.read_csv(data_dir / "Label.csv")

print(f"Data shape:   {df.shape}")
print(f"Labels shape: {labels.shape}")
print(f"\nLabel column name: {labels.columns.tolist()}")
print(f"Unique label values: {sorted(labels['Label'].unique())}")

# Attach label to features
df['Label'] = labels['Label'].values

print(f"\nCombined shape: {df.shape}")
print(f"\nFirst 3 rows:")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../data/UNSW-NB15/UNSW-NB15_features.csv'

In [ ]:
# Map integer labels to attack category names
# UNSW-NB15 label mapping (0-9)
label_map = {
    0: 'Normal',
    1: 'Fuzzers',
    2: 'Analysis',
    3: 'Backdoors',
    4: 'DoS',
    5: 'Exploits',
    6: 'Generic',
    7: 'Reconnaissance',
    8: 'Shellcode',
    9: 'Worms'
}

df['attack_cat'] = df['Label'].map(label_map)

# Sanity check — flag any labels that didn't map
unmapped = df['attack_cat'].isnull().sum()
if unmapped:
    print(f"WARNING: {unmapped} rows have unmapped labels")
    print(df[df['attack_cat'].isnull()]['Label'].value_counts())
else:
    print("All labels mapped successfully")

In [ ]:
# Per-feature statistics
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Label']  # exclude the raw label

stats = df[numeric_cols].describe().T
stats['% null'] = (df[numeric_cols].isnull().mean() * 100).round(2)
stats = stats[['min', 'max', 'mean', 'std', '% null']]

print(stats)
stats.to_csv("../data/eda_unswnb15_stats.csv")

In [ ]:
# Class label distribution
cat_counts = df['attack_cat'].value_counts()
pcts = (cat_counts / len(df) * 100).round(2)

print("Attack category counts:")
print(cat_counts)
print("\nPercentages:")
print(pcts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
cat_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('UNSW-NB15 — Attack Category Counts')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Percentage plot
pcts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('UNSW-NB15 — Attack Category %')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('% of dataset')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("../data/eda_unswnb15_label_dist.png", dpi=150)
plt.show()

In [ ]:
# Constant features (zero variance)
constant_features = [col for col in numeric_cols if df[col].nunique() <= 1]

print(f"Constant features ({len(constant_features)}):")
for col in constant_features:
    print(f"  '{col}'  —  value: {df[col].unique()}")

In [ ]:
# Duplicate rows
n_dupes = df.duplicated().sum()
pct_dupes = (n_dupes / len(df) * 100).round(2)

print(f"Duplicate rows: {n_dupes} ({pct_dupes}%)")

if n_dupes > 0:
    print("\nSample of duplicates:")
    display(df[df.duplicated(keep=False)].head(10))

In [ ]:
# Infinite values
inf_counts = np.isinf(df[numeric_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]

print(f"Columns with infinite values ({len(inf_cols)}):")
print(inf_cols if len(inf_cols) else "None")